# Long-short agent portfolio from prediction market insights

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/crowdvector/polybridge-cookbooks/blob/main/longshort-portfolio/longshort-portfolio.ipynb)

This notebook explains the Claude Desktop + PolyBridge MCP setup and then runs the reproducible REST-backed portfolio sizing helper. It maps live Forecast probabilities into a constrained scoring rule, a position table, and order instructions.

PolyBridge MCP release: https://github.com/crowdvector/polybridge-search-mcp/releases/tag/polybridge-mcp-v0.2.4


In [ ]:
%pip install --quiet requests pillow


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/crowdvector/polybridge-cookbooks.git"
REPO_ROOT = Path("/content/polybridge-cookbooks")

def resolve_workdir() -> Path:
    candidates = [
        Path.cwd(),
        Path.cwd() / "longshort-portfolio",
        Path.cwd().parent,
    ]
    for candidate in candidates:
        helper = candidate / "portfolio_sizing.py"
        if helper.exists():
            return helper.parent

    helper = REPO_ROOT / "longshort-portfolio" / "portfolio_sizing.py"
    if helper.exists():
        return helper.parent

    try:
        subprocess.run(
            ["git", "clone", REPO_URL, str(REPO_ROOT)],
            check=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
        )
    except subprocess.CalledProcessError as exc:
        raise FileNotFoundError(
            "portfolio_sizing.py is not present locally. The fallback clone path uses the public repo."
        ) from exc

    helper = REPO_ROOT / "longshort-portfolio" / "portfolio_sizing.py"
    if not helper.exists():
        raise FileNotFoundError("portfolio_sizing.py was not found after cloning the public repo.")
    return helper.parent

workdir = resolve_workdir()
os.chdir(workdir)
if str(workdir) not in sys.path:
    sys.path.insert(0, str(workdir))

import portfolio_sizing
workdir


## Standardized Claude/MCP Prompt

The helper below uses REST for reproducibility. For the interactive workflow, use the exact prompt stored in `PROMPT.md`. If `POLYBRIDGE_API_KEY` is missing from the environment, notebook mode falls back to `getpass()` inside the helper instead of printing or persisting the key. The notebook keeps API key handling local and never prints or persists the key.


In [ ]:
from IPython.display import Markdown, display

display(Markdown((workdir / "PROMPT.md").read_text(encoding="utf-8")))


In [ ]:
paths, bundle = portfolio_sizing.run_once(prompt_for_key=True)
paths.as_dict()


In [ ]:
from IPython.display import JSON, Image

display(Markdown("## Macro Snapshot"))
display(JSON(bundle["macro_snapshot"]))
display(Markdown("## Position Table"))
display(JSON(bundle["position_table"]))
display(Markdown("## Order Instructions"))
display(JSON(bundle["order_instructions"]))

summary_png = paths.portfolio_summary_png
if summary_png is not None and summary_png.exists():
    display(Markdown("## Portfolio Summary Image"))
    display(Image(filename=str(summary_png)))


In [ ]:
display(Markdown("## Research Brief"))
display(Markdown(bundle["research_brief"]))
display(Markdown("## Agent Session"))
display(Markdown(bundle["agent_session"]))
